# Dataset class counts by patient ID

This notebook reads `lung_us.yml`, iterates over the dataset images/labels,
and computes the number of YOLO class instances per patient ID.

Patient IDs are inferred from the beginning of each image filename,
using the same convention as `kfold_dataset_split.py` (prefix before the first underscore).

In [15]:
from pathlib import Path
from collections import defaultdict

import yaml
import pandas as pd

# Path to the dataset YAML used for training
data_yaml_path = Path('lung_us.yml').resolve()
data_yaml_path

WindowsPath('D:/Repos/aigt/UltrasoundObjectDetection/YOLOv8/lung_us.yml')

In [16]:
# Load YAML and resolve base dataset path and image directories
with data_yaml_path.open('r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

base_path = Path(data_cfg['path']).expanduser()
train = data_cfg.get('train')
val = data_cfg.get('val')
test = data_cfg.get('test')
names = data_cfg.get('names', {})

image_dirs = []
for sub in (train, val, test):
    if isinstance(sub, str) and sub:
        image_dirs.append(base_path / sub)

image_dirs

[WindowsPath('D:/Data/ObjectDetection/all/train/images')]

In [17]:
# Helper to infer patient ID from image filename
def patient_id_from_image_path(image_path: Path) -> str:
    """Extract patient ID from image filename.

    Assumes filenames start with the patient ID followed by an underscore,
    e.g. 'SCN22_frame001.png' -> patient ID 'SCN22'.
    """
    stem = image_path.stem
    return stem.split('_')[0]


IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}


def find_label_for_image(image_path: Path) -> Path:
    """Return the expected YOLO label path for a given image.

    Assumes this fixed layout:
    - Images are under a directory named 'images'
    - Labels are under a sibling directory named 'labels'
    - Label filename is identical to the image stem with '.txt' extension

    Example:
    D:/Data/ObjectDetection/all/train/images/SCN1_frame001.png
    -> D:/Data/ObjectDetection/all/train/labels/SCN1_frame001.txt
    """
    parts = list(image_path.parts)

    # Find the last occurrence of 'images' and replace it with 'labels'
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] == 'images':
            parts[i] = 'labels'
            labels_dir = Path(*parts).parent  # directory .../labels
            return labels_dir / (image_path.stem + '.txt')

    # If 'images' is not present for some reason, fall back to same directory
    return image_path.with_suffix('.txt')


find_label_for_image(Path('D:/Data/ObjectDetection/all/train/images/SCN1_frame001.png'))

WindowsPath('D:/Data/ObjectDetection/all/train/labels/SCN1_frame001.txt')

In [22]:
# Iterate over all images and accumulate class counts per patient ID
patient_class_counts = defaultdict(lambda: defaultdict(int))
total_missing_label_files = 0
total_images = 0

for img_dir in image_dirs:
    if not img_dir.exists():
        continue
    for p in img_dir.rglob('*'):
        if not p.is_file():
            continue
        if p.suffix.lower() not in IMG_EXTENSIONS:
            continue

        total_images += 1
        pid = patient_id_from_image_path(p)
        label_path = find_label_for_image(p)

        if not label_path.exists():
            total_missing_label_files += 1
            continue

        with label_path.open('r', encoding='utf-8') as lf:
            for line in lf:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                try:
                    cls = int(parts[0])
                except (ValueError, IndexError):
                    continue
                patient_class_counts[pid][cls] += 1

total_images, total_missing_label_files


(1097, 6)

In [23]:
# Convert to a DataFrame: rows = patients, columns = class IDs (and totals)
all_patient_ids = sorted(patient_class_counts.keys())
all_class_ids = sorted({c for counts in patient_class_counts.values() for c in counts.keys()})

rows = []
for pid in all_patient_ids:
    row = {'patient_id': pid}
    total_for_patient = 0
    for cid in all_class_ids:
        count = patient_class_counts[pid].get(cid, 0)
        row[f'class_{cid}'] = count
        total_for_patient += count
    row['total_instances'] = total_for_patient
    rows.append(row)

# Build DataFrame; handle the case where there are no rows
if rows:
    df = pd.DataFrame(rows)
    df = df.sort_values(by='total_instances', ascending=False).reset_index(drop=True)
else:
    # Empty DataFrame with the expected columns
    df = pd.DataFrame(columns=['patient_id', 'total_instances'])

df

,patient_id,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,total_instances
0,SCN2,455,789,124,355,15,0,0,0,1738
1,SCN1,313,529,48,135,380,257,0,29,1691
2,SCN17,52,724,4,233,50,211,72,14,1360
3,SCN22,383,0,0,0,336,28,0,0,747


In [24]:
# Optional: map class column names to human-readable labels from the YAML
rename_map = {}
for col in df.columns:
    if col.startswith('class_') and col[len('class_'):].isdigit():
        cid = int(col[len('class_'):])
        class_name = names.get(cid, None)
        if class_name is not None:
            rename_map[col] = f'{col}__{class_name}'

df_renamed = df.rename(columns=rename_map)
df_renamed


,patient_id,class_0__Pleura-intact,class_1__Pleura-affected,class_2__B-lines,class_3__B-lines-septal or glass rockets,class_4__A-lines,class_5__Consolidation-nontranslobar,class_6__Consolidation-translobar,class_7__Pleural effusion,total_instances
0,SCN2,455,789,124,355,15,0,0,0,1738
1,SCN1,313,529,48,135,380,257,0,29,1691
2,SCN17,52,724,4,233,50,211,72,14,1360
3,SCN22,383,0,0,0,336,28,0,0,747
